# ShowDiffraction

Diffraction-pattern **d-spacing** analysis for a single 2D pattern (SAED) or a 3D
stack. Find the beam center, pick Bragg **spots** and **rings**, read calibrated
d-spacings, and calibrate k-space from a known reflection. For 4D-STEM, use
`Show4DSTEM`.


In [1]:
import numpy as np

rng = np.random.default_rng(0)
size = 256
c = (size - 1) / 2
yy, xx = np.mgrid[0:size, 0:size]
r = np.sqrt((yy - c) ** 2 + (xx - c) ** 2)

# Single-crystal SAED: a square Bragg lattice of Gaussian spots + central beam
dp_sc = np.zeros((size, size), np.float32)
for h in range(-4, 5):
    for k in range(-4, 5):
        pr, pc = c + h * 28.0, c + k * 28.0
        amp = 6.0 if (h == 0 and k == 0) else 1.0 / (1 + 0.4 * (h * h + k * k))
        dp_sc += amp * np.exp(-0.5 * (((yy - pr) ** 2 + (xx - pc) ** 2) / 2.0 ** 2))

# Polycrystalline: Debye-Scherrer rings + central beam
dp_poly = 6.0 * np.exp(-0.5 * (r / 5.0) ** 2)
for rad, amp in [(34, 1.0), (58, 0.7), (80, 0.5), (104, 0.35)]:
    dp_poly += amp * np.exp(-0.5 * ((r - rad) / 2.0) ** 2)
dp_poly = (dp_poly + 0.01 * rng.random((size, size))).astype(np.float32)


## Single-crystal SAED

Pass the beam center, auto-detect Bragg spots, and read calibrated d-spacings.


In [2]:
from quantem.widget import ShowDiffraction

w = ShowDiffraction(
    dp_sc, center=(c, c), bf_radius=14, k_pixel_size=0.12,
    title="Single-crystal SAED", offline=True,
)
w.detect_spots(max_spots=12)
w


  to cpu: 0.01s (0.3 MB)


ShowDiffraction(shape=(1, 256, 256), sampling=(1.0 Å, 0.12 1/Å), frame=0/1, spots=12, title='Single-crystal SAED')

## Polycrystalline rings

Auto-detect Debye–Scherrer rings, then calibrate k-space from a ring of known
d-spacing.


In [3]:
ring = ShowDiffraction(dp_poly, title="Polycrystalline", offline=True)
ring.detect_rings(max_rings=4)
if ring.rings:
    ring.calibrate_from_ring(ring.rings[0]["radius_px"], d_known=2.34)  # e.g. Au {111}
ring


  to cpu: 0.03s (0.3 MB)


ShowDiffraction(shape=(1, 256, 256), sampling=(1.0 Å, 0.007409958764211522 1/Å), frame=0/1, title='Polycrystalline')

## Save as a standalone HTML

`export_html` writes a self-contained file that opens in any browser without a
Jupyter kernel.


In [4]:
path = w.export_html("showdiffraction_saed.html")
print("wrote", path.name)


wrote showdiffraction_saed.html
